In [2]:
%pip install mysql-connector-python

   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   - -------------------------------------- 0.8/17.7 MB 3.8 MB/s eta 0:00:05
   ---- ----------------------------------- 2.1/17.7 MB 5.7 MB/s eta 0:00:03
   ----- ---------------------------------- 2.4/17.7 MB 5.6 MB/s eta 0:00:03
   ----- ---------------------------------- 2.4/17.7 MB 5.6 MB/s eta 0:00:03
   ----- ---------------------------------- 2.4/17.7 MB 5.6 MB/s eta 0:00:03
   ----- ---------------------------------- 2.6/17.7 MB 2.3 MB/s eta 0:00:07
   ------ --------------------------------- 2.9/17.7 MB 2.0 MB/s eta 0:00:08
   ------- -------------------------------- 3.1/17.7 MB 1.9 MB/s eta 0:00:08
   -------- ------------------------------- 3.9/17.7 MB 2.0 MB/s eta 0:00:07
   ----------- ---------------------------- 5.0/17.7 MB 2.3 MB/s eta 0:00:06
   -------------- ------------------------- 6.6/17.7 MB 2.7 MB/s eta 0:00:05
   ----------

In [3]:
import mysql.connector

print("MySQL connector ready")

MySQL connector ready


In [4]:
import pandas as pd
import requests

In [1]:
import mysql.connector

print("Testing connection...")

Testing connection...


In [1]:
%pip show mysql-connector-python

Name: mysql-connector-python
Version: 26.7.0
Summary: A self-contained Python driver for communicating with MySQL servers, using an API that is compliant with the Python Database API Specification v2.0 (PEP 249).
Home-page: https://dev.mysql.com/doc/connector-python/en/
Author: Oracle and/or its affiliates
Author-email: 
License: GNU GPLv2 (with FOSS License Exception)
Location: c:\Users\chamm\AppData\Local\Programs\Python\Python313\Lib\site-packages
Requires: 
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [2]:
import mysql.connector
print("Connector imported successfully")

Connector imported successfully


In [3]:
import pandas as pd

cities = pd.read_csv("top_20_cities.csv")

cities

,customer_city,customer_state,customer_records
0,sao paulo,SP,15540
1,rio de janeiro,RJ,6882
2,belo horizonte,MG,2773
3,brasilia,DF,2131
4,curitiba,PR,1521
5,campinas,SP,1444
6,porto alegre,RS,1379
7,salvador,BA,1245
8,guarulhos,SP,1189
9,sao bernardo do campo,SP,938


In [4]:
import requests
import pandas as pd
import time

def geocode_city(city, state):
    url = "https://geocoding-api.open-meteo.com/v1/search"
    
    params = {
        "name": f"{city}, {state}",
        "count": 1,
        "language": "en",
        "format": "json",
        "countryCode": "BR"
    }
    
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    
    results = response.json().get("results", [])
    
    if not results:
        return pd.Series([None, None, None])
    
    result = results[0]
    
    return pd.Series([
        result.get("name"),
        result.get("latitude"),
        result.get("longitude")
    ])

cities[["geocoded_name", "latitude", "longitude"]] = cities.apply(
    lambda row: geocode_city(row["customer_city"], row["customer_state"]),
    axis=1
)

cities

,customer_city,customer_state,customer_records,geocoded_name,latitude,longitude
0,sao paulo,SP,15540,São Paulo,-23.54750,-46.63611
1,rio de janeiro,RJ,6882,Rio de Janeiro,-22.90642,-43.18223
2,belo horizonte,MG,2773,Belo Horizonte,-19.92083,-43.93778
3,brasilia,DF,2131,Brasília,-15.77972,-47.92972
4,curitiba,PR,1521,Curitiba,-25.42778,-49.27306
5,campinas,SP,1444,Campinas,-22.90556,-47.06083
6,porto alegre,RS,1379,Porto Alegre,-30.03283,-51.23019
7,salvador,BA,1245,Salvador,-12.97563,-38.49096
8,guarulhos,SP,1189,Guarulhos,-23.46278,-46.53333
9,sao bernardo do campo,SP,938,São Bernardo do Campo,-23.69389,-46.56500


In [5]:
cities.to_csv("top_20_cities_geocoded.csv", index=False)

print("Saved:", len(cities), "cities")
cities.head()

Saved: 20 cities


,customer_city,customer_state,customer_records,geocoded_name,latitude,longitude
0,sao paulo,SP,15540,São Paulo,-23.54750,-46.63611
1,rio de janeiro,RJ,6882,Rio de Janeiro,-22.90642,-43.18223
2,belo horizonte,MG,2773,Belo Horizonte,-19.92083,-43.93778
3,brasilia,DF,2131,Brasília,-15.77972,-47.92972
4,curitiba,PR,1521,Curitiba,-25.42778,-49.27306


In [6]:
import requests

test_city = cities.iloc[0]

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": test_city["latitude"],
    "longitude": test_city["longitude"],
    "start_date": "2018-01-01",
    "end_date": "2018-01-07",
    "daily": [
        "temperature_2m_mean",
        "precipitation_sum",
        "wind_speed_10m_max"
    ],
    "timezone": "America/Sao_Paulo"
}

response = requests.get(url, params=params, timeout=30)

print("Status code:", response.status_code)

weather_test = response.json()

print(weather_test.keys())
print(weather_test["daily"])

Status code: 200
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
{'time': ['2018-01-01', '2018-01-02', '2018-01-03', '2018-01-04', '2018-01-05', '2018-01-06', '2018-01-07'], 'temperature_2m_mean': [22.5, 22.2, 21.2, 20.9, 20.3, 21.3, 18.2], 'precipitation_sum': [1.6, 1.1, 3.4, 0.0, 3.0, 9.4, 12.0], 'wind_speed_10m_max': [20.2, 28.8, 24.5, 20.6, 17.7, 16.6, 15.8]}


In [7]:
weather_test_df = pd.DataFrame(weather_test["daily"])

weather_test_df

,time,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,2018-01-01,22.5,1.6,20.2
1,2018-01-02,22.2,1.1,28.8
2,2018-01-03,21.2,3.4,24.5
3,2018-01-04,20.9,0.0,20.6
4,2018-01-05,20.3,3.0,17.7
5,2018-01-06,21.3,9.4,16.6
6,2018-01-07,18.2,12.0,15.8


In [8]:
all_weather = []

for _, city in cities.iterrows():
    print(f"Fetching: {city['customer_city']}")

    params = {
        "latitude": city["latitude"],
        "longitude": city["longitude"],
        "start_date": "2017-01-01",
        "end_date": "2018-10-31",
        "daily": [
            "temperature_2m_mean",
            "precipitation_sum",
            "wind_speed_10m_max"
        ],
        "timezone": "America/Sao_Paulo"
    }

    response = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()["daily"]

    weather_city = pd.DataFrame(data)
    weather_city["customer_city"] = city["customer_city"]
    weather_city["customer_state"] = city["customer_state"]

    all_weather.append(weather_city)

    time.sleep(0.5)

weather_df = pd.concat(all_weather, ignore_index=True)

print("\nWeather dataset shape:", weather_df.shape)
weather_df.head()

Fetching: sao paulo
Fetching: rio de janeiro
Fetching: belo horizonte
Fetching: brasilia
Fetching: curitiba
Fetching: campinas
Fetching: porto alegre
Fetching: salvador
Fetching: guarulhos
Fetching: sao bernardo do campo
Fetching: niteroi
Fetching: santo andre
Fetching: osasco
Fetching: santos
Fetching: goiania
Fetching: sao jose dos campos
Fetching: fortaleza
Fetching: sorocaba
Fetching: recife
Fetching: florianopolis

Weather dataset shape: (13380, 6)


,time,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,customer_city,customer_state
0,2017-01-01,24.8,2.0,21.3,sao paulo,SP
1,2017-01-02,23.9,5.1,16.0,sao paulo,SP
2,2017-01-03,23.1,5.0,15.2,sao paulo,SP
3,2017-01-04,23.5,7.3,28.3,sao paulo,SP
4,2017-01-05,24.7,3.0,19.0,sao paulo,SP


In [9]:
weather_df.to_csv("weather_20_cities_2017_2018.csv", index=False)

print("Saved successfully.")
print("Rows:", len(weather_df))
print("Cities:", weather_df["customer_city"].nunique())
print("Date range:", weather_df["time"].min(), "to", weather_df["time"].max())

Saved successfully.
Rows: 13380
Cities: 20
Date range: 2017-01-01 to 2018-10-31


In [10]:
orders = pd.read_csv(
    "../02_data/raw/olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

customers = pd.read_csv(
    "../02_data/raw/olist_customers_dataset.csv"
)

print("Orders:", orders.shape)
print("Customers:", customers.shape)

Orders: (99441, 8)
Customers: (99441, 5)


In [11]:
orders_weather = orders.merge(
    customers[["customer_id", "customer_city", "customer_state"]],
    on="customer_id",
    how="left"
)

print("Shape:", orders_weather.shape)
print("Missing cities:", orders_weather["customer_city"].isna().sum())

orders_weather[
    ["order_id", "order_purchase_timestamp",
     "order_delivered_customer_date",
     "order_estimated_delivery_date",
     "customer_city", "customer_state"]
].head()

Shape: (99441, 10)
Missing cities: 0


,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,santo andre,SP


In [12]:
weather_orders = orders_weather[
    (orders_weather["order_status"] == "delivered") &
    (orders_weather["order_delivered_customer_date"].notna()) &
    (orders_weather["order_estimated_delivery_date"].notna()) &
    (orders_weather["customer_city"].isin(cities["customer_city"]))
].copy()

print("Orders available for weather analysis:", len(weather_orders))
print("Cities covered:", weather_orders["customer_city"].nunique())

Orders available for weather analysis: 40622
Cities covered: 20


In [13]:
weather_orders["delivery_delay_days"] = (
    weather_orders["order_delivered_customer_date"]
    - weather_orders["order_estimated_delivery_date"]
).dt.days

weather_orders["delivery_status"] = weather_orders["delivery_delay_days"].apply(
    lambda x: "Late" if x > 0 else "On Time / Early"
)

print(weather_orders["delivery_delay_days"].describe())

print("\nDelivery status:")
print(weather_orders["delivery_status"].value_counts())

count    40622.000000
mean       -11.349860
std          9.924076
min        -83.000000
25%        -16.000000
50%        -11.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64

Delivery status:
delivery_status
On Time / Early    37845
Late                2777
Name: count, dtype: int64


In [14]:
weather_df["date"] = pd.to_datetime(weather_df["time"])

weather_orders["delivery_date"] = (
    weather_orders["order_delivered_customer_date"].dt.normalize()
)

weather_df = weather_df.rename(columns={
    "customer_city": "weather_city",
    "customer_state": "weather_state"
})

print(weather_df[["date", "weather_city", "weather_state",
                  "precipitation_sum", "wind_speed_10m_max"]].head())

        date weather_city weather_state  precipitation_sum  wind_speed_10m_max
0 2017-01-01    sao paulo            SP                2.0                21.3
1 2017-01-02    sao paulo            SP                5.1                16.0
2 2017-01-03    sao paulo            SP                5.0                15.2
3 2017-01-04    sao paulo            SP                7.3                28.3
4 2017-01-05    sao paulo            SP                3.0                19.0


In [15]:
weather_analysis = weather_orders.merge(
    weather_df[
        [
            "date",
            "weather_city",
            "weather_state",
            "temperature_2m_mean",
            "precipitation_sum",
            "wind_speed_10m_max"
        ]
    ],
    left_on=["delivery_date", "customer_city", "customer_state"],
    right_on=["date", "weather_city", "weather_state"],
    how="left"
)

print("Shape:", weather_analysis.shape)

print(
    "\nMissing weather:",
    weather_analysis["precipitation_sum"].isna().sum()
)

weather_analysis[
    [
        "order_id",
        "customer_city",
        "delivery_date",
        "delivery_delay_days",
        "temperature_2m_mean",
        "precipitation_sum",
        "wind_speed_10m_max"
    ]
].head()

Shape: (40622, 19)

Missing weather: 114


,order_id,customer_city,delivery_date,delivery_delay_days,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,e481f51cbdc54678b7cc49136f2d6af7,sao paulo,2017-10-10,-8,23.0,2.4,16.9
1,ad21c59c0840e6cb83a9ceb5573f8159,santo andre,2018-02-16,-10,18.6,0.8,22.6
2,e69bfb5eb88e0ed6a785585b27e16dbf,sorocaba,2017-08-16,-7,16.4,7.3,12.8
3,e6ce16cb79ec1d90b1da9085a6118aeb,rio de janeiro,2017-05-29,-9,21.1,0.0,9.4
4,34513ce0c4fab462a55830c0989c7edb,sao paulo,2017-07-19,-20,10.3,0.1,16.6


In [16]:
missing_weather = weather_analysis[
    weather_analysis["precipitation_sum"].isna()
].copy()

print("Missing weather rows:", len(missing_weather))

print("\nCities with missing weather:")
print(
    missing_weather["customer_city"]
    .value_counts()
)

print("\nMissing weather dates:")
print(
    missing_weather["delivery_date"]
    .sort_values()
    .head(20)
)

Missing weather rows: 114

Cities with missing weather:
customer_city
sao paulo                31
rio de janeiro           27
belo horizonte           10
brasilia                  6
campinas                  6
curitiba                  6
recife                    4
santo andre               4
fortaleza                 4
goiania                   4
sao bernardo do campo     3
santos                    2
porto alegre              2
osasco                    1
guarulhos                 1
niteroi                   1
sao jose dos campos       1
sorocaba                  1
Name: count, dtype: int64

Missing weather dates:
536     2016-10-11
29665   2016-10-13
23444   2016-10-13
35236   2016-10-13
27630   2016-10-13
36115   2016-10-13
3022    2016-10-14
10007   2016-10-14
2569    2016-10-14
8774    2016-10-14
37657   2016-10-14
39984   2016-10-14
3483    2016-10-15
39404   2016-10-15
31919   2016-10-15
24411   2016-10-15
39728   2016-10-15
5949    2016-10-15
15344   2016-10-15
2539    2016-10

In [17]:
print(
    weather_orders["delivery_date"].min()
)

2016-10-11 00:00:00


In [18]:
weather_2016 = []

for _, city in cities.iterrows():
    print(f"Fetching 2016: {city['customer_city']}")

    params = {
        "latitude": city["latitude"],
        "longitude": city["longitude"],
        "start_date": "2016-10-11",
        "end_date": "2016-12-31",
        "daily": [
            "temperature_2m_mean",
            "precipitation_sum",
            "wind_speed_10m_max"
        ],
        "timezone": "America/Sao_Paulo"
    }

    response = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()["daily"]

    city_weather = pd.DataFrame(data)
    city_weather["customer_city"] = city["customer_city"]
    city_weather["customer_state"] = city["customer_state"]

    weather_2016.append(city_weather)

    time.sleep(0.5)

weather_2016_df = pd.concat(weather_2016, ignore_index=True)

print("\n2016 weather shape:", weather_2016_df.shape)
print(
    "Date range:",
    weather_2016_df["time"].min(),
    "to",
    weather_2016_df["time"].max()
)

Fetching 2016: sao paulo
Fetching 2016: rio de janeiro
Fetching 2016: belo horizonte
Fetching 2016: brasilia
Fetching 2016: curitiba
Fetching 2016: campinas
Fetching 2016: porto alegre
Fetching 2016: salvador
Fetching 2016: guarulhos
Fetching 2016: sao bernardo do campo
Fetching 2016: niteroi
Fetching 2016: santo andre
Fetching 2016: osasco
Fetching 2016: santos
Fetching 2016: goiania
Fetching 2016: sao jose dos campos
Fetching 2016: fortaleza
Fetching 2016: sorocaba
Fetching 2016: recife
Fetching 2016: florianopolis

2016 weather shape: (1640, 6)
Date range: 2016-10-11 to 2016-12-31


In [19]:
weather_2016_df["date"] = pd.to_datetime(weather_2016_df["time"])

weather_all = pd.concat(
    [weather_df, weather_2016_df],
    ignore_index=True
)

print("Combined weather shape:", weather_all.shape)
print(
    "Date range:",
    weather_all["date"].min(),
    "to",
    weather_all["date"].max()
)

Combined weather shape: (15020, 9)
Date range: 2016-10-11 00:00:00 to 2018-10-31 00:00:00


In [20]:
weather_2016_df = weather_2016_df.rename(columns={
    "customer_city": "weather_city",
    "customer_state": "weather_state"
})

weather_all = pd.concat(
    [weather_df, weather_2016_df],
    ignore_index=True
)

print("Shape:", weather_all.shape)
print("Columns:", weather_all.columns.tolist())
print("Missing city:", weather_all["weather_city"].isna().sum())
print("Missing state:", weather_all["weather_state"].isna().sum())

Shape: (15020, 7)
Columns: ['time', 'temperature_2m_mean', 'precipitation_sum', 'wind_speed_10m_max', 'weather_city', 'weather_state', 'date']
Missing city: 0
Missing state: 0


In [21]:
weather_analysis = weather_orders.merge(
    weather_all[
        [
            "date",
            "weather_city",
            "weather_state",
            "temperature_2m_mean",
            "precipitation_sum",
            "wind_speed_10m_max"
        ]
    ],
    left_on=[
        "delivery_date",
        "customer_city",
        "customer_state"
    ],
    right_on=[
        "date",
        "weather_city",
        "weather_state"
    ],
    how="left"
)

print("Shape:", weather_analysis.shape)
print("Missing weather:", weather_analysis["precipitation_sum"].isna().sum())
print("Missing temperature:", weather_analysis["temperature_2m_mean"].isna().sum())
print("Missing wind:", weather_analysis["wind_speed_10m_max"].isna().sum())

Shape: (40622, 19)
Missing weather: 1
Missing temperature: 1
Missing wind: 1


In [22]:
missing_weather = weather_analysis[
    weather_analysis["precipitation_sum"].isna()
][[
    "order_id",
    "customer_city",
    "customer_state",
    "delivery_date",
    "delivery_delay_days",
    "delivery_status"
]]

missing_weather

,order_id,customer_city,customer_state,delivery_date,delivery_delay_days,delivery_status
993,9a194607b0125b5c6c0e01a53843080f,santo andre,PB,2018-02-16,-21,On Time / Early


In [23]:
weather_analysis_clean = weather_analysis.dropna(
    subset=[
        "temperature_2m_mean",
        "precipitation_sum",
        "wind_speed_10m_max"
    ]
).copy()

print("Original observations:", len(weather_analysis))
print("Clean observations:", len(weather_analysis_clean))
print("Removed:", len(weather_analysis) - len(weather_analysis_clean))

Original observations: 40622
Clean observations: 40621
Removed: 1


In [24]:
weather_analysis_clean["weather_condition"] = (
    weather_analysis_clean["precipitation_sum"]
    .gt(0)
    .map({True: "Rainy", False: "No Rain"})
)

weather_summary = (
    weather_analysis_clean
    .groupby("weather_condition")
    .agg(
        orders=("order_id", "count"),
        late_orders=("delivery_status", lambda x: (x == "Late").sum()),
        avg_delay_days=("delivery_delay_days", "mean"),
        avg_precipitation=("precipitation_sum", "mean")
    )
    .reset_index()
)

weather_summary["late_rate"] = (
    weather_summary["late_orders"]
    / weather_summary["orders"]
    * 100
)

weather_summary

,weather_condition,orders,late_orders,avg_delay_days,avg_precipitation,late_rate
0,No Rain,15355,904,-11.565418,0.000000,5.887333
1,Rainy,25266,1873,-11.218475,4.470629,7.413124


In [25]:
weather_analysis_clean["rain_level"] = pd.cut(
    weather_analysis_clean["precipitation_sum"],
    bins=[-0.01, 0, 5, float("inf")],
    labels=["No Rain", "Light Rain (0–5 mm)", "Heavy Rain (>5 mm)"]
)

rain_intensity_summary = (
    weather_analysis_clean
    .groupby("rain_level", observed=True)
    .agg(
        orders=("order_id", "count"),
        late_orders=("delivery_status", lambda x: (x == "Late").sum()),
        avg_delay_days=("delivery_delay_days", "mean")
    )
    .reset_index()
)

rain_intensity_summary["late_rate"] = (
    rain_intensity_summary["late_orders"]
    / rain_intensity_summary["orders"]
    * 100
)

rain_intensity_summary

,rain_level,orders,late_orders,avg_delay_days,late_rate
0,No Rain,15355,904,-11.565418,5.887333
1,Light Rain (0–5 mm),18523,1354,-11.167036,7.309831
2,Heavy Rain (>5 mm),6743,519,-11.359781,7.696871
